In [ ]:
#@title Setup Repository and Install Dependencies

import os
import subprocess
import sys
import time
from google.colab import drive
from google.colab import userdata
drive.mount('/content/drive')

time.sleep(2)

github_token = userdata.get('GITHUB_TOKEN')

DRIVE_DIR = '/content/drive/MyDrive/Qwen_SDXL_Trainer'
REPO_DIR = os.path.join(DRIVE_DIR, 'qwen-sdxl-adapter')
ADAPTER_DIR = os.path.join(DRIVE_DIR, 'checkpoints/phase2_diff2flow')
OUTPUT_DIR = os.path.join(DRIVE_DIR, 'outputs')

REPO_URL_STR = 'https://github.com/cosmicoxytocin/qwen-sdxl-adapter.git'
REPO_URL = REPO_URL_STR.replace('https://', f'https://{github_token}@')
REPO_BRANCH = 'revert-inf' #@param {type: 'string'}

UPDATE_REPO = True #@param {type: 'boolean'}

os.makedirs(OUTPUT_DIR, exist_ok=True)

#@markdown **Note:** The first time you run this cell, the runtime will crash. **THIS IS OKAY!** It will automatically reconnect.<br> Re-run this cell after the session crashes.

#%pip install -q -U transformers diffusers accelerate safetensors "pillow==10.2.0"
try:
    import PIL
    current_pil = PIL.__version__
except ImportError:
    current_pil = None
if current_pil != '10.2.0':
    print("Installing dependencies...")
    subprocess.run('pip install -q -U transformers diffusers accelerate safetensors "pillow==10.2.0"', shell=True)
    print("Dependencies installed successfully.")
    time.sleep(2)
    print("Stopping RUNTIME. Please run this cell again.")
    os.kill(os.getpid(), 9)
    


def run_cmd(cmd, cwd=None):
    result = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error running '{cmd}': \n{result.stderr}")
    return result.stdout.strip()

def setup_repo():
    if not os.path.exists(os.path.join(REPO_DIR, '.git')):
        print(f"Cloning repository from {REPO_URL_STR}...")
        os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)
        run_cmd(f"git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
        print("Repository cloned successfully.")
    else:
        current_branch = run_cmd("git branch --show-current", cwd=REPO_DIR)
        if current_branch != REPO_BRANCH:
            print(f"Switching to branch '{REPO_BRANCH}'...")
            run_cmd("git fetch --all", cwd=REPO_DIR)
            run_cmd(f"git checkout {REPO_BRANCH}", cwd=REPO_DIR)
            run_cmd(f"git pull origin {REPO_BRANCH}", cwd=REPO_DIR)
            print(f"Switched to branch '{REPO_BRANCH}' and updated.")
        elif UPDATE_REPO:
            print("Updating repository...")
            output = run_cmd(f"git pull origin {REPO_BRANCH}", cwd=REPO_DIR)
            if "Already up to date." in output:
                print("Repository is already up to date.")
            else:
                print("Repository updated successfully.")
        else:
            print("Skipping repository update.")

print("Setting up repository...")
os.chdir(DRIVE_DIR)
setup_repo()
print("Repository setup complete.")

time.sleep(2)

%cd {REPO_DIR}

In [ ]:
#@title Load Model into VRAM
import torch
from scripts.inference import QwenSDXLPipeline

ADAPTER_NAME = "adapter_step_4000.safetensors" #@param {type: 'string'}
ADAPTER_PATH = os.path.join(ADAPTER_DIR, ADAPTER_NAME)

torch.cuda.empty_cache()

print(f"Loading pipeline with adapter '{ADAPTER_NAME}'...")
pipe = QwenSDXLPipeline(
    adapter_ckpt_path=ADAPTER_PATH,
    device='cuda'
)
print("Pipeline loaded successfully.")

In [ ]:
#@title Inference

prompt = "" #@param {type: 'string'}
negative_prompt = "" #@param {type: 'string'}

#@markdown ---

width = 1024 # @param {type:"slider", min:512, max:1536, step:64}
height = 1024 # @param {type:"slider", min:512, max:1536, step:64}

#@markdown ---


steps = 25 #@param {type:'slider', min:1, max:100, step:1}
cfg_scale = 4.5 #@param {type:'slider', min:1.0, max:10.0, step:0.5}

#@markdown ---

seed = 42 #@param {type:'number'}

#@markdown ---

print("Generating image...")
img = pipe.generate(
    prompt=prompt,
    negative_prompt=negative_prompt,
    width=width,
    height=height,
    num_inference_steps=steps,
    guidance_scale=cfg_scale,
    seed=seed if seed != -1 else None
)

display(img)

save_path = os.path.join(OUTPUT_DIR, f"output_{int(time.time())}.png")
img.save(save_path)
print(f"Image saved to {save_path}")